In [ ]:
import pandas as pd
import pickle
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

deliveries = pd.read_csv('data/deliveries.csv')
matches = pd.read_csv('data/matches.csv')

# Step 1 — Aggregate per over
overdata = deliveries.groupby(
    ['match_id', 'inning', 'batting_team', 'bowling_team', 'over']
).agg(
    runs_in_over=('total_runs', 'sum'),
    wickets_in_over=('is_wicket', 'sum')
).reset_index()

# Step 2 — Sort and calculate cumulative stats
overdata = overdata.sort_values(['match_id', 'inning', 'over'])

overdata['cumulative_runs'] = overdata.groupby(
    ['match_id', 'inning']
)['runs_in_over'].cumsum()

overdata['cumulative_wickets'] = overdata.groupby(
    ['match_id', 'inning']
)['wickets_in_over'].cumsum()

# Step 3 — Add final score
final_scores = overdata.groupby(
    ['match_id', 'inning']
)['cumulative_runs'].max().reset_index()
final_scores.columns = ['match_id', 'inning', 'final_score']
overdata = overdata.merge(final_scores, on=['match_id', 'inning'])

# Step 4 — Add venue from matches.csv
matches_venue = matches[['id', 'venue']].rename(columns={'id': 'match_id'})
overdata = overdata.merge(matches_venue, on='match_id')

# Step 5 — Feature engineering
overdata['current_run_rate'] = (
    overdata['cumulative_runs'] / overdata['over']
).replace([float('inf'), float('-inf')], 0)

overdata['wickets_remaining'] = 10 - overdata['cumulative_wickets']
overdata['overs_remaining'] = 20 - overdata['over']

# Fix 1 — Separate innings 1 and 2
overdata_1 = overdata[overdata['inning'] == 1].copy()

# Fix 2 — Only predict from over 6 onwards
overdata_1 = overdata_1[overdata_1['over'] >= 6]

# Step 6 — Encode teams and venue
le = LabelEncoder()
overdata_1['batting_team'] = le.fit_transform(overdata_1['batting_team'])
overdata_1['bowling_team'] = le.fit_transform(overdata_1['bowling_team'])
overdata_1['venue'] = le.fit_transform(overdata_1['venue'])

# Step 7 — Features and target
features = [
    'over',
    'cumulative_runs',
    'cumulative_wickets',
    'current_run_rate',
    'wickets_remaining',
    'overs_remaining',
    'batting_team',
    'bowling_team',
    'venue'              # new feature added
]

X = overdata_1[features]
y = overdata_1['final_score']

# Step 8 — Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Fix 3 — Better model settings
model = RandomForestRegressor(
    n_estimators=200,        # more trees
    max_depth=15,            # deeper trees
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1                # use all CPU cores = faster
)

model.fit(X_train, y_train)
print("Model trained successfully!")

predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
print(f"MAE: {mae:.1f} runs")

# Bonus — see which features matter most

with open ('scorepredictormodel.pkl','wb') as f:
    pickle.dump(model,f)
    
print("Score prediction model saved!")  

Model trained successfully!
MAE: 10.1 runs
